In [ ]:
from helperfunctions import helper as hfn
from helperfunctions import training_lib as tl
from helperfunctions import intern_constants as ic
import matplotlib.pyplot as plt
from matplotlib.ticker import  FormatStrFormatter, MaxNLocator, AutoMinorLocator
from typing import Optional
import pandas as pd
import numpy as np
import torch.nn as nn
from glob import glob
import os
from pathlib import Path
import pandas as pd
#%matplotlib widget


In [ ]:
fn_train = "train_eval_df.csv"
fn_val1 = "val1_eval_df.csv"
fn_val2  = "val2_eval_df.csv"

In [ ]:
path  = ic.PATH_PART2_EVAL
os.makedirs(path, exist_ok=True)
if not Path(path /"train_eval_df.csv").exists():
    cfg1 = hfn.TrainConfig(config_name="kde", choose_val_set=1)
    cfg2 = hfn.TrainConfig(config_name="kde", choose_val_set=2)
    print(f"val1 start: {cfg1.val_start_time}, end:{cfg1.val_end_time}")
    print(f"val2 start: {cfg2.val_start_time}, end:{cfg2.val_end_time}")
    ae_best = tl.get_model_results(ic.PATH_TO_BEST_MODEL_DIR, best_n=1)
    ae, _,_,_,_ = ae_best[0]
    ae = ae.to(cfg1.device).eval()
    print(ae)
    
    train_loader, val1_loader, _ = hfn.build_dataloaders(
    train_csv_dir=ic.PATH_PC_FILTERING,
    val_csv_dir=ic.PATH_IMPUTED,
    test_csv_dir=ic.PATH_IMPUTED,
    cfg=cfg1
    )
    train_eval_df = tl.eval_model(
    model=ae,
    data_loader=train_loader,
    device=cfg1,
    loss_fn = nn.MSELoss(reduction="none")
    )
    train_loader = None
    train_eval_df.to_csv(path /fn_train, index=False)
    
    val1_eval_df = tl.eval_model(
    model=ae,
    data_loader=val1_loader,
    device=cfg1,
    loss_fn = nn.MSELoss(reduction="none"),
    )
    val1_loader = None
    val1_eval_df.to_csv(path/fn_val1, index=False)
    
    _, val2_loader, _ = hfn.build_dataloaders(
    train_csv_dir=ic.PATH_PC_FILTERING,
    val_csv_dir=ic.PATH_IMPUTED,
    test_csv_dir=ic.PATH_IMPUTED,
    cfg=cfg2)
    
    val2_eval_df = tl.eval_model(
    model=ae,
    data_loader=val2_loader,
    device=cfg2,
    loss_fn = nn.MSELoss(reduction="none"),
    )
    val2_loader = None
    val2_eval_df.to_csv(path/fn_val2, index=False)

else:
    train_eval_df = pd.read_csv(path/fn_train)
    val1_eval_df = pd.read_csv(path/fn_val1)
    val2_eval_df = pd.read_csv(path/fn_val2)



In [ ]:
fs = glob(os.path.join(ic.PATH_PART2_EVAL_TEST, "*.csv"))
test_eval_df = pd.read_csv(fs[0])

### KDE Plot per Turbine
Model output RE (MSE)<br>
Used Kernel: Gaussian<br>
bandwith-factor for 1d: variance of the kernel covariance<br>

In [ ]:
display(train_eval_df.head())

In [ ]:
wts = [1,2,4,5,6,7,8,9,10,11,12,13,14,15]
cmap = plt.get_cmap("tab20", len(wts))
color_map = {wt: cmap(i) for i, wt in enumerate(wts)}

In [ ]:

def print_kdes(df:pd.DataFrame,
               wts: list[int] = [1,2,4,5,6,7,8,9,10,11,12,13,14,15],
               ncols: int = 1,
               title: str = "",
               dpi:int=300,
               filename:Optional[str] = None,
               xlim: Optional[float] = None,
               show_global_max = False,
               all_in_one = False,
               ):
    cmap = plt.get_cmap("tab20", len(wts))
    color_map = {wt: cmap(i) for i, wt in enumerate(wts)}
    
    if not all_in_one:
        n = len(wts)
        nrows = (n + ncols - 1) // ncols
        
        fig, axes = plt.subplots(nrows, ncols, figsize=(16, 3.5*nrows), sharex=False, sharey=False)
        axes = np.array(axes).ravel()
        global_max = pd.to_numeric(df[ic.MEAN_LOSS_PER_SAMPLE]).max()
        for i, wt in enumerate(wts):
            ax = axes[i]
            wt_df = df[df[ic.WT_ID] == wt]
            re = pd.to_numeric(wt_df[ic.MEAN_LOSS_PER_SAMPLE])
            
            x_max = re.max() if xlim is None else xlim
            
            re.plot.kde(ax=ax, label=f"WT {wt}", color=color_map[wt]) #bandwith method "scott"
            ax.set_xlim(0.0, global_max if show_global_max else x_max)
            ax.set_xlabel("Reconstruction Error")
            ax.set_ylabel("Density")
            ax.set_title(f"KDE WT:{wt}")
            ax.grid(True, alpha=0.3)
            ax.legend(fontsize=9)

        for j in range(i+1, len(axes)):
            axes[j].axis("off")
        
        major = 10
        minor = 5
        
        for ax in axes:
            if ax.axison:
                ax.xaxis.set_major_locator(MaxNLocator(nbins=major, prune=None))
                ax.xaxis.set_minor_locator(AutoMinorLocator(minor))
                ax.xaxis.set_minor_formatter(FormatStrFormatter("%.2f"))
                ax.tick_params(axis="x", which="minor", labelsize=7, pad=2)
                ax.tick_params(axis="x", which="major", labelsize=7)
                
                #ax.tick_params(axis="x", which="minor", labelbottom=False)
                #ax.set_xlabel("Reconstruction Error")
        fig.suptitle(f"{title}", y=1.02)
        
    else:
        fig,ax = plt.subplots(figsize=(8,4))
        global_max= 0.0
        for wt in wts:
            wt_df = train_eval_df[train_eval_df[ic.WT_ID] == wt]
            re = pd.to_numeric(wt_df[ic.MEAN_LOSS_PER_SAMPLE])
            global_max = max(global_max, re.max())    
            re.plot.kde(ax=ax, label=f"WT {wt}", color=color_map[wt]) #bandwith method "scott"

        ax.set_xlim(0.0, global_max if show_global_max else xlim if xlim is not None else 0.04)
        ax.set_xlabel("Reconstruction Error")
        ax.set_ylabel("Density")
        ax.set_title(f"KDE All WTs - {title}")
        ax.grid(True, alpha=0.3)
        ax.legend(ncol=2, fontsize=9)
        plt.tight_layout()
        plt.show()
    plt.tight_layout()
    if (filename is not None):
            fig.savefig(ic.PATH_PRINTS / filename, 
                        dpi = dpi, 
                        bbox_inches="tight"
                        )
    plt.show()

In [ ]:
print_kdes(train_eval_df, title="KDE of REs per WT (Train Set)")

In [ ]:
print_kdes(train_eval_df, title="KDE of REs per WT (Train Set)", all_in_one=True, show_global_max=True)

In [ ]:
print_kdes(train_eval_df, title="KDE of REs per WT (Train Set)", all_in_one=True, show_global_max=False, xlim=0.04)

In [ ]:
print_kdes(val1_eval_df, title="KDE of REs per WT (Val1 Set - used for early stopping)")

In [ ]:
print_kdes(val1_eval_df, title="KDE of REs per WT (Val1 Set - used for early stopping)", all_in_one=True, show_global_max=True)
print_kdes(val1_eval_df, title="KDE of REs per WT (Val1 Set - used for early stopping)", all_in_one=True, show_global_max=False, xlim=0.04)

In [ ]:
print_kdes(val2_eval_df, title="KDE of REs per WT (Val2 Set - used for anomaly injection)")

In [ ]:
print_kdes(val2_eval_df, title="KDE of REs per WT (Val2 Set - used for anomaly injection)", all_in_one=True, show_global_max=True)
print_kdes(val2_eval_df, title="KDE of REs per WT (Val2 Set - used for anomaly injection)", all_in_one=True, show_global_max=False, xlim=0.04)

In [ ]:
print_kdes(test_eval_df, title="KDE of REs per WT (Test Set)")

In [ ]:
print_kdes(test_eval_df, title="KDE of REs per WT (Test Set)", all_in_one=True, show_global_max=True)
print_kdes(test_eval_df, title="KDE of REs per WT (Test Set)", all_in_one=True, show_global_max=False, xlim=0.04)